# Window C — Peer Review Run (Google Colab)

**Purpose:** let a reviewer reproduce the A3 baseline first, then optionally test one changed assumption on their own copy.

## Reviewer instructions
1. Open this notebook in Google Colab from the GitHub link.
2. Choose **File → Save a copy in Drive**. Work only in your copy; the shared GitHub original is not changed.
3. Run the notebook once with `RUN_MODE = "baseline"` and do not change the parameters. Use **Runtime → Run all**.
4. Confirm: 36 scenario rows, 25/25 baseline internal checks, and the GeoPandas S2 / east / w=1.50 m overlay result of 0.30 m².
5. Optional: in the **EDIT PARAMETERS HERE** cell, change one parameter only, set `RUN_MODE = "experiment"`, then choose **Runtime → Run all** again. Compare the new CSV/figures with the baseline.

**Where to edit:** Cell 2, under `EDIT PARAMETERS HERE`. Do not edit the analysis-function cell. For a controlled test, change only one of `corridor_half_widths_m`, `band_inner_offset_c1_m`, `band_outer_offset_c2_m`, or `assumed_band_threshold_pct` at a time. Keep exactly three corridor-width values if you want to preserve the 36-scenario grid.

**Important:** the 25/25 figure applies to the unchanged A3 baseline. In experiment mode, the notebook runs general structure and independent interval checks instead of the baseline-specific known-value assertions. A changed experiment must not be reported as the original 25/25 baseline.

- Design scenario: one 2 m × 1 m seat; six alternative placements in an assumed 6 m × 6 m Window C.
- Baseline scenario grid: 6 placements × 3 corridor widths × 2 assumed edge proxies = 36 rows.
- Geometry is a local-metre schematic, not georeferenced Hong Kong data.
- East / North edges are assumed proxies, not observed pedestrian routes.
- c₁, c₂ and the 60% threshold are researcher assumptions, not validated comfort standards.
- This workflow screens geometric relationships; it does not measure pedestrian flow, dwell time or actual short-stay comfort.

**Outputs:** CSV files and PNG figures are saved in the Colab `outputs/` folder. Download them using the Files panel on the left before the temporary Colab runtime ends.


In [ ]:
# 0 · Install/check the libraries used by the notebook.
# Colab runs on a temporary cloud machine; no local Python installation is required.
import importlib.util, subprocess, sys, platform
required = {
    "numpy": "numpy==2.3.5", "pandas": "pandas==2.2.3",
    "shapely": "shapely==2.1.2", "geopandas": "geopandas==1.1.2",
    "pyproj": "pyproj==3.7.2", "matplotlib": "matplotlib==3.10.8",
    "requests": "requests==2.32.5",
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
else:
    print("Required libraries are already available.")
print("Python:", platform.python_version())


In [ ]:
# 1 · EDIT PARAMETERS HERE — this is the only cell reviewers need to change.
# First run: keep RUN_MODE="baseline" and all values unchanged.
# Optional test: change ONE parameter, set RUN_MODE="experiment", and Run all again.

from pathlib import Path
import json

RUN_MODE = "baseline"  # change to "experiment" only after the baseline run

PARAMETERS = {
    "window_width_m": 6.0,
    "window_height_m": 6.0,
    "seat_length_m": 2.0,
    "seat_depth_m": 1.0,
    "corridor_half_widths_m": [0.75, 1.0, 1.5],  # keep 3 values for 36 scenarios
    "flow_edge_proxies": ["east", "north"],
    "band_inner_offset_c1_m": 0.30,
    "band_outer_offset_c2_m": 1.50,
    "assumed_band_threshold_pct": 60.0,
    "base_half_width_m": 1.0,
    "base_flow_edge_proxy": "east",
    "near_north_side_m": 3.0,
}

# Fixed project metadata and the six original A3 alternatives.
CONFIG = {
    "study": "A3 Window C / Short-stay seating geometry prototype",
    "coordinate_frame": "LOCAL_METRIC_SCHEMATIC_NOT_A_GEOSPATIAL_CRS",
    **PARAMETERS,
    "osm_reference_way_id": 27059126,
    "osm_northeast_node_id": 296894103,
    "osm_southeast_node_id": 296894102,
    "osm_window_start_along_east_edge_m": 5.0,
    "osm_data_downloaded_in_this_package": False,
    "external_osm_fetch_optional": True,
    "site_validation_status": "PENDING",
}

VARIANTS_CSV = """variant,position,orientation,center_from_east_m,center_from_north_m
S1,near,parallel,2.2,3.0
S2,near,perpendicular,2.2,3.0
S3,centre,parallel,3.2,3.0
S4,centre,perpendicular,3.2,3.0
S5,far,parallel,4.3,3.0
S6,far,perpendicular,4.3,3.0
"""

ROOT = Path.cwd()
(ROOT / "inputs").mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)
(ROOT / "inputs" / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
(ROOT / "inputs" / "variants.csv").write_text(VARIANTS_CSV, encoding="utf-8")

if RUN_MODE not in {"baseline", "experiment"}:
    raise ValueError("RUN_MODE must be 'baseline' or 'experiment'")
if len(PARAMETERS["corridor_half_widths_m"]) != 3:
    raise ValueError("Keep exactly three corridor widths to preserve the 36-scenario comparison.")
if PARAMETERS["band_outer_offset_c2_m"] <= PARAMETERS["band_inner_offset_c1_m"]:
    raise ValueError("c2 must be greater than c1.")

print("Mode:", RUN_MODE)
print("Editable parameters:", json.dumps(PARAMETERS, indent=2))
print("Project folder:", ROOT)


### 2 · Inspectable analysis functions

The following code cell contains a direct copy of the repository's `a3src/a3_workflow.py` **functions**, so the Colab version can run without cloning a repository. This same module is the canonical local Python script. It constructs hypothetical 6×6 m geometry, applies Shapely/GeoPandas overlays, saves CSV/PNG outputs and records checks. No real local street geometry is loaded.


In [ ]:
"""Re-runnable A3 Window C short-stay *geometry* prototype.

Coordinate convention (NOT EPSG:2326):
    x = -u, where u >= 0 metres inward from the assumed east boundary.
    y = -v, where v >= 0 metres southward from the assumed north boundary.
    Window C is -6 <= x <= 0, -6 <= y <= 0.

This geometry is researcher-defined: true OSM coordinates and local walking
routes are NOT packaged. Code correctness does not prove short-stay comfort.
"""

from __future__ import annotations

import csv
import json
import math
import platform
from pathlib import Path
import sys
from typing import Any

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch, Rectangle
import numpy as np
import pandas as pd
import pyproj
import shapely
from shapely.affinity import translate
from shapely.geometry import LineString, Polygon, box, mapping
from shapely.ops import transform

GREEN = "#225d51"
DARK = "#263f3b"
MUTED = "#607c70"
PALE = "#e7eee8"
ORANGE = "#c65c39"
GOLD = "#cbab64"


def load_inputs(project_root: Path):
    project_root = Path(project_root)
    cfg = json.loads((project_root / "inputs" / "config.json").read_text(encoding="utf-8"))
    variants = pd.read_csv(project_root / "inputs" / "variants.csv")
    required = {"variant", "position", "orientation", "center_from_east_m", "center_from_north_m"}
    if not required.issubset(variants.columns):
        raise ValueError(f"variants.csv lacks columns: {sorted(required - set(variants.columns))}")
    if variants.variant.nunique() != 6:
        raise ValueError("A3 prototype expects exactly six distinct seat variants")
    if not cfg["band_outer_offset_c2_m"] > cfg["band_inner_offset_c1_m"] >= 0:
        raise ValueError("Must satisfy 0 <= c1 < c2")
    if cfg["window_width_m"] <= 0 or cfg["window_height_m"] <= 0:
        raise ValueError("Window dimensions must be positive")
    return cfg, variants


def window_polygon(cfg: dict) -> Polygon:
    return box(-float(cfg["window_width_m"]), -float(cfg["window_height_m"]), 0.0, 0.0)


def seat_polygon(row: pd.Series, cfg: dict) -> Polygon:
    """Parallel to EAST edge is longitudinal N-S, not E-W."""
    x = -float(row.center_from_east_m)
    y = -float(row.center_from_north_m)
    L, D = float(cfg["seat_length_m"]), float(cfg["seat_depth_m"])
    orientation = str(row.orientation).lower()
    if orientation == "parallel":
        width_x, height_y = D, L
    elif orientation == "perpendicular":
        width_x, height_y = L, D
    else:
        raise ValueError(f"Unknown orientation: {orientation}")
    poly = box(x-width_x/2, y-height_y/2, x+width_x/2, y+height_y/2)
    if not window_polygon(cfg).covers(poly):
        raise ValueError(f"Seat {row.variant} is outside Window C")
    return poly


def proxy_line(cfg: dict, edge: str, offset_m: float = 0) -> LineString:
    """An assumed EDGE as flow proxy, not an observed pedestrian route."""
    W, H = float(cfg["window_width_m"]), float(cfg["window_height_m"])
    if edge == "east":
        return LineString([(offset_m, 0), (offset_m, -H)])
    if edge == "north":
        return LineString([(-W, offset_m), (0, offset_m)])
    raise ValueError(f"Unknown edge proxy {edge}")


def overlay_metrics(seat: Polygon, cfg: dict, edge: str, w: float, offset_m: float = 0) -> dict:
    """GeoPandas/Shapely operations in local metres: corridor and band overlays."""
    if w <= 0:
        raise ValueError("w must be positive")
    c1 = float(cfg["band_inner_offset_c1_m"])
    c2 = float(cfg["band_outer_offset_c2_m"])
    win = window_polygon(cfg)
    line = proxy_line(cfg, edge, offset_m)
    # The proxy edge is intentionally not an observed walking centreline.
    corridor = line.buffer(w, cap_style="flat").intersection(win)
    band = line.buffer(w+c2, cap_style="flat").difference(
        line.buffer(w+c1, cap_style="flat")
    ).intersection(win)
    overlap = float(seat.intersection(corridor).area)
    band_m2 = float(seat.intersection(band).area)
    seat_m2 = float(seat.area)
    return {
        "corridor_overlap_m2": round(overlap, 6),
        "band_area_m2": round(band_m2, 6),
        "band_share_pct": round(100*band_m2/seat_m2, 6),
        "seat_area_m2": round(seat_m2, 6),
        "corridor": corridor,
        "band": band,
    }


def generate_scenarios(cfg: dict, variants: pd.DataFrame) -> pd.DataFrame:
    """Six seat variants × three w assumptions × two flow proxies = 36 records."""
    rows = []
    target = float(cfg["assumed_band_threshold_pct"])
    for _, v in variants.iterrows():
        seat = seat_polygon(v, cfg)
        for edge in cfg["flow_edge_proxies"]:
            for w in cfg["corridor_half_widths_m"]:
                m = overlay_metrics(seat, cfg, edge, float(w))
                conflict = m["corridor_overlap_m2"] > 1e-8
                proxy_pass = m["band_share_pct"] + 1e-8 >= target
                status = ("GEOMETRIC_CONFLICT" if conflict else
                          "ASSUMED_PROXY_PASS" if proxy_pass else
                          "BELOW_ASSUMED_PROXY_THRESHOLD")
                rows.append({
                    "variant": v.variant,
                    "position": v.position,
                    "orientation": v.orientation,
                    "centre_u_from_east_m": v.center_from_east_m,
                    "centre_v_from_north_m": v.center_from_north_m,
                    "flow_edge_proxy": edge,
                    "corridor_half_width_w_m": float(w),
                    "seat_area_m2": m["seat_area_m2"],
                    "corridor_overlap_m2": m["corridor_overlap_m2"],
                    "proposed_band_share_pct": m["band_share_pct"],
                    "assumed_threshold_pct": target,
                    "scenario_status": status,
                    "source_level": "SCHEMATIC_ASSUMPTION_NOT_FIELD_VALIDATED",
                })
    frame = pd.DataFrame(rows).sort_values(["variant","flow_edge_proxy","corridor_half_width_w_m"]).reset_index(drop=True)
    expected = len(variants) * len(cfg["corridor_half_widths_m"]) * len(cfg["flow_edge_proxies"])
    assert len(frame) == expected, f"Expected {expected} rows, got {len(frame)}"
    return frame


def manual_band_pct_east(row: pd.Series, cfg: dict, w: float) -> float:
    """Independent 1D interval arithmetic check, no Shapely calls."""
    u = float(row.center_from_east_m)
    L, D = float(cfg["seat_length_m"]), float(cfg["seat_depth_m"])
    uextent = D if row.orientation == "parallel" else L
    lower_seat, upper_seat = u - uextent/2, u + uextent/2
    lower_band = w + float(cfg["band_inner_offset_c1_m"])
    upper_band = w + float(cfg["band_outer_offset_c2_m"])
    len_overlap = max(0.0, min(upper_seat, upper_band) - max(lower_seat, lower_band))
    return 100 * len_overlap / uextent


def verification_checks(cfg: dict, variants: pd.DataFrame, results: pd.DataFrame) -> pd.DataFrame:
    def get(variant, edge, w, field):
        a=results[(results.variant==variant)&(results.flow_edge_proxy==edge)&
                  (np.isclose(results.corridor_half_width_w_m,w))]
        if len(a)!=1:
            raise AssertionError(f"Expected one row for {variant}/{edge}/{w}, got {len(a)}")
        return float(a.iloc[0][field])
    checks=[]
    def record(name, expected, observed, tol=1e-8):
        ok=bool(abs(observed-expected)<=tol)
        checks.append({"check":name,"expected":expected,"observed":observed,"status":"PASS" if ok else "FAIL"})
        if not ok:
            raise AssertionError(f"{name}: expected {expected}, saw {observed}")
    record("Scenario row count",36,float(len(results)))
    record("All seat areas = 2 m2", 2.0, float(results.seat_area_m2.min()))
    record("Seat area maximum = 2 m2", 2.0, float(results.seat_area_m2.max()))
    record("S1 east base band share % (analytical 0.8 * 2 / 2)", 80.0, get("S1","east",1.0,"proposed_band_share_pct"))
    record("S1 east w=0.75 band share %",55.0,get("S1","east",0.75,"proposed_band_share_pct"))
    record("S2 east w=1.0 band share %",60.0,get("S2","east",1.0,"proposed_band_share_pct"))
    record("S2 east w=1.5 corridor overlap m2",0.30,get("S2","east",1.5,"corridor_overlap_m2"))
    for _, v in variants.iterrows():
        for w in cfg["corridor_half_widths_m"]:
            observed=get(v.variant,"east",float(w),"proposed_band_share_pct")
            hand=manual_band_pct_east(v,cfg,float(w))
            record(f"Independent interval check {v.variant}, east w={w}",hand,observed,tol=1e-6)
    return pd.DataFrame(checks)


def summarize_variants(results: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for variant, group in results.groupby("variant"):
        east=group[group.flow_edge_proxy=="east"]
        base=east[np.isclose(east.corridor_half_width_w_m,1.0)]
        east_ok=bool((east.scenario_status == "ASSUMED_PROXY_PASS").all())
        every_ok=bool((group.scenario_status == "ASSUMED_PROXY_PASS").all())
        rows.append({
            "variant": variant,
            "base_east_share_pct":float(base.proposed_band_share_pct.iloc[0]),
            "min_east_share_pct":float(east.proposed_band_share_pct.min()),
            "max_east_overlap_m2":float(east.corridor_overlap_m2.max()),
            "east_all_three_widths_pass":east_ok,
            "all_six_cases_both_edges_pass":every_ok,
        })
    return pd.DataFrame(rows)


def _draw_polygon(ax, poly, color: str, alpha=0.25, ec=None, lw=1):
    if poly.is_empty:
        return
    geoms=list(poly.geoms) if hasattr(poly, "geoms") else [poly]
    for p in geoms:
        if p.is_empty or not isinstance(p, Polygon): continue
        xs,ys=p.exterior.xy
        ax.fill(xs,ys,facecolor=color,edgecolor=ec or color,alpha=alpha,linewidth=lw)


def plot_six_placements(cfg: dict, variants: pd.DataFrame, path: Path):
    fig, axes=plt.subplots(2,3,figsize=(13,7.6),sharex=True,sharey=True)
    w=float(cfg["base_half_width_m"])
    for ax,(_,v) in zip(axes.ravel(),variants.iterrows()):
        poly=seat_polygon(v,cfg)
        m=overlay_metrics(poly,cfg,"east",w)
        _draw_polygon(ax,m["corridor"],"#dcbd9d",0.55)
        _draw_polygon(ax,m["band"],"#b8cfbf",0.65)
        _draw_polygon(ax,poly,ORANGE,0.68,ec=ORANGE,lw=1.5)
        ax.plot([0,0],[0,-6],color=GREEN,lw=2.5)
        ax.plot([-6,0,0,-6,-6],[0,0,-6,-6,0],color=MUTED,linestyle="--",lw=0.9)
        pct=m["band_share_pct"]
        overlap=m["corridor_overlap_m2"]
        ax.set_title(f"{v.variant} · {v.position} / {v.orientation}\nBand share {pct:.0f}% · overlap {overlap:.2f} m²",fontsize=10,color=DARK)
        ax.set_xlim(-6.5,.45);ax.set_ylim(-6.5,.4);ax.set_aspect("equal",adjustable="box")
        ax.grid(True,alpha=.10)
        ax.set_xlabel("Local x (m) · east edge at x=0",fontsize=8)
        ax.set_ylabel("Local y (m) · north edge at y=0",fontsize=8)
    fig.suptitle("Window C / six ALTERNATIVE placements of ONE 2 × 1 m seat\nAssumed 6 × 6 m geometry; east edge proxy, w = 1.0 m",fontsize=14,weight="bold",color=GREEN,y=.995)
    fig.text(.5,.01,"Schematic geometry only • green = proposed flow-adjacent band • tan = assumed corridor • orange = seat",ha="center",fontsize=9,color=MUTED)
    fig.tight_layout(rect=[0,.04,1,.91]);fig.savefig(path,dpi=170,bbox_inches="tight",facecolor="white");plt.close(fig)


def plot_comparison(cfg: dict, results: pd.DataFrame, path: Path):
    base=results[(results.flow_edge_proxy=="east") &
                 (np.isclose(results.corridor_half_width_w_m,cfg["base_half_width_m"]))].sort_values("variant")
    fig, ax=plt.subplots(figsize=(10.8,5.3))
    vals=base.proposed_band_share_pct.to_numpy()
    names=base.variant.to_list()
    cols=[GREEN if v>=cfg["assumed_band_threshold_pct"] and ov<=0 else MUTED
          for v,ov in zip(vals,base.corridor_overlap_m2)]
    bars=ax.bar(names,vals,color=cols,width=.66)
    s=cfg["assumed_band_threshold_pct"]
    ax.axhline(s,color=ORANGE,lw=2,linestyle="--",label=f"Assumed research threshold s = {s:.0f}%")
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,v+2,f"{v:.0f}%",ha="center",fontsize=11,color=DARK)
    ax.set_ylim(0,105);ax.set_ylabel("Proposed flow-adjacent band share (%)")
    ax.set_title("Base-case S1 leads; proxy threshold is NOT a comfort standard",color=GREEN,weight="bold",fontsize=13)
    ax.text(.5,-.15,"East boundary proxy · corridor half-width w = 1.00 m · c₁=0.30m, c₂=1.50m",ha="center",transform=ax.transAxes,color=MUTED,fontsize=10)
    ax.legend(frameon=False,loc="upper right")
    ax.spines[['top','right']].set_visible(False)
    fig.tight_layout();fig.savefig(path,dpi=190,bbox_inches="tight",facecolor="white");plt.close(fig)


def _eval_at_w(cfg,variants,variant,w,edge="east"):
    v=variants[variants.variant==variant].iloc[0]
    return overlay_metrics(seat_polygon(v,cfg),cfg,edge,float(w))


def plot_width_sensitivity(cfg:dict,variants:pd.DataFrame,path:Path):
    # Recompute the feasible interval from the CURRENT parameters; do not hard-code
    # the baseline 0.80–1.70 m limits when a reviewer edits the parameter cell.
    ws=np.linspace(.5,2.1,321)
    fig,ax=plt.subplots(figsize=(10.8,5.5))
    s1_pass=None
    for vname,color in [("S1",GREEN),("S2",ORANGE)]:
        shares=np.array([_eval_at_w(cfg,variants,vname,float(w))["band_share_pct"] for w in ws])
        overlaps=np.array([_eval_at_w(cfg,variants,vname,float(w))["corridor_overlap_m2"] for w in ws])
        ax.plot(ws,shares,label=f"{vname} band share",lw=2.5,color=color)
        ax.plot(ws[overlaps>1e-8],shares[overlaps>1e-8],linestyle="None",marker="x",color=color,markersize=3,alpha=.55)
        for w in cfg["corridor_half_widths_m"]:
            m=_eval_at_w(cfg,variants,vname,w)
            ax.scatter([w],[m["band_share_pct"]],c=[color],s=38,zorder=6)
        if vname=="S1":
            s1_pass=(shares+1e-8>=float(cfg["assumed_band_threshold_pct"])) & (overlaps<=1e-8)
    threshold=float(cfg["assumed_band_threshold_pct"])
    ax.axhline(threshold,color=ORANGE,ls="--",lw=1.7,label=f"Assumed s = {threshold:.0f}%")
    # Shade each contiguous sampled interval where S1 meets BOTH geometric rules.
    idx=np.where(s1_pass)[0]
    ranges=[]
    if len(idx):
        groups=np.split(idx,np.where(np.diff(idx)>1)[0]+1)
        for group in groups:
            lo,hi=float(ws[group[0]]),float(ws[group[-1]])
            ranges.append((lo,hi))
            ax.axvspan(lo,hi,color=GREEN,alpha=.09)
            ax.axvline(lo,color=GOLD,ls=":",lw=1.2)
            ax.axvline(hi,color=GOLD,ls=":",lw=1.2)
    ax.set_xlim(.5,2.1);ax.set_ylim(0,108)
    ax.set_xlabel("Assumed corridor half-width w (m)");ax.set_ylabel("Proposed band share (%)")
    if ranges:
        range_text="; ".join([f"{a:.2f}–{b:.2f} m" for a,b in ranges])
        title=f"S1 conditional feasible w range (east proxy): {range_text}"
        ax.text(.52,7,"Shaded ranges also require zero corridor overlap",color=GREEN,fontsize=8)
    else:
        title="S1 has no sampled w range satisfying both rules (east proxy)"
    ax.set_title(title,color=GREEN,weight="bold",fontsize=11)
    ax.legend(loc="upper left",frameon=False)
    ax.spines[['top','right']].set_visible(False);fig.tight_layout();fig.savefig(path,dpi=190,bbox_inches="tight",facecolor="white");plt.close(fig)


def position_sweep(cfg:dict,variants:pd.DataFrame)->pd.DataFrame:
    """East-proxy-only exploratory search; does not prove real site suitability."""
    row=variants[variants.variant=="S1"].iloc[0].copy()
    rows=[]
    for u in np.round(np.arange(1.5,4.501,0.05),2):
        row["center_from_east_m"]=float(u)
        # Centres too near edges create invalid geometries, filter with flag.
        try: seat=seat_polygon(row,cfg)
        except ValueError: continue
        ms=[overlay_metrics(seat,cfg,"east",w) for w in cfg["corridor_half_widths_m"]]
        clear=all(m["corridor_overlap_m2"]<1e-8 for m in ms)
        minshare=min(m["band_share_pct"] for m in ms)
        rows.append({"centre_u_m":u,"min_east_band_share_pct":minshare,
                     "max_east_overlap_m2":max(m["corridor_overlap_m2"] for m in ms),
                     "passes_all_east_widths_assumed":clear and minshare+1e-8>=cfg["assumed_band_threshold_pct"]})
    return pd.DataFrame(rows)


def run_project(project_root:Path,create_plots=True,perform_checks=True)->dict[str,Any]:
    project_root=Path(project_root)
    outdir=project_root/"outputs"
    outdir.mkdir(exist_ok=True,parents=True)
    cfg, variants=load_inputs(project_root)
    df=generate_scenarios(cfg,variants)
    summary=summarize_variants(df)
    sweep=position_sweep(cfg,variants)
    if perform_checks:
        checks=verification_checks(cfg,variants,df)
    else:checks=pd.DataFrame()
    df.to_csv(outdir/"a3_scenarios.csv",index=False,float_format="%.6f")
    summary.to_csv(outdir/"a3_variant_summary.csv",index=False,float_format="%.6f")
    sweep.to_csv(outdir/"a3_position_sweep_east.csv",index=False,float_format="%.6f")
    checks.to_csv(outdir/"a3_verification_internal.csv",index=False,float_format="%.6f")
    # Local metric WKT, not GeoJSON: no CRS is misrepresented as WGS84 or EPSG:2326.
    with (outdir/"a3_seat_geometry_LOCAL_METRES.wkt.csv").open("w",newline="",encoding="utf-8") as f:
        writer=csv.writer(f);writer.writerow(["variant","geometry_wkt","coordinate_frame"])
        for _,v in variants.iterrows():
            writer.writerow([v.variant,seat_polygon(v,cfg).wkt,cfg["coordinate_frame"]])
    if create_plots:
        plot_six_placements(cfg,variants,outdir/"a3_six_alternative_placements.png")
        plot_comparison(cfg,df,outdir/"a3_base_comparison.png")
        plot_width_sensitivity(cfg,variants,outdir/"a3_w_sensitivity.png")
    provenance={
        "generated_by":"a3_workflow.run_project",
        "coordinate_frame":cfg["coordinate_frame"],
        "source_of_site_outline":"OSM Way ID only; actual OSM coordinates not loaded in offline geometric prototype",
        "flow_proxy_validation":"PENDING",
        "behavioural_short_stay_validation":"PENDING",
        "unverified": ["Actual walking route and route level","Window C field dimensions","c1/c2/s from behavioural evidence","Human QGIS independent check"],
        "python":sys.version.split()[0],
        "geopandas":gpd.__version__,"shapely":shapely.__version__,
        "pyproj":pyproj.__version__,"pandas":pd.__version__,"matplotlib":matplotlib.__version__,
    }
    (outdir/"a3_run_record.json").write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding="utf-8")
    return {"scenarios":df,"summary":summary,"sweep":sweep,"verification":checks,"run_record":provenance}


def osm_fetch_optional(project_root:Path, *,do_fetch=False,timeout_seconds=25)->dict:
    """OPTIONAL: verify real OSM NE/SE nodes; skip by default so offline rerun works.

    If enabled and internet is available, download the named OSM way and
    compute NE->SE EPSG:2326 distance. This does not validate a walking path.
    """
    from datetime import datetime,timezone
    project_root=Path(project_root)
    cfg,_=load_inputs(project_root)
    record={"attempted":False,"status":"NOT_FETCHED", "reason":"Offline reproducibility default"}
    if not do_fetch:return record
    import requests
    import xml.etree.ElementTree as ET
    url=f"https://api.openstreetmap.org/api/0.6/way/{cfg['osm_reference_way_id']}/full"
    record={"attempted":True,"url":url,"utc_time":datetime.now(timezone.utc).isoformat()}
    try:
        response=requests.get(url,timeout=timeout_seconds,headers={"User-Agent":"ARCH7476-A3-Research-Notebook/1.0"})
        response.raise_for_status()
        xml=response.text
        tree=ET.fromstring(xml)
        nodes={int(n.attrib["id"]):(float(n.attrib["lon"]),float(n.attrib["lat"])) for n in tree.findall("node")}
        refs=[int(n.attrib["ref"]) for n in tree.find("way").findall("nd")]
        ne,se=cfg["osm_northeast_node_id"],cfg["osm_southeast_node_id"]
        if ne not in nodes or se not in nodes or ne not in refs or se not in refs:
            raise ValueError("Expected NE/SE node IDs not found on returned OSM way. Review source assumptions.")
        transformer=pyproj.Transformer.from_crs("EPSG:4326","EPSG:2326",always_xy=True)
        east=transformer.transform(*nodes[ne]);south=transformer.transform(*nodes[se])
        observed_length=math.dist(east,south)
        record.update(status="OSM_NODES_FETCHED",east_length_m=round(observed_length,4),
                      node_NE_4326=nodes[ne],node_SE_4326=nodes[se],
                      note="Boundary length only; not a field-measured path or observed flow.")
        (project_root/"inputs"/"osm_way_27059126_full.xml").write_text(xml,encoding="utf-8")
        # Optional geographic anchoring: original OSM points saved as WGS84 GeoJSON.
        gj={"type":"FeatureCollection","features":[
            {"type":"Feature","properties":{"way_id":cfg["osm_reference_way_id"],"role":"east-boundary-reference", "not_an_observed_pedestrian_path":True},
             "geometry":{"type":"LineString","coordinates":[list(nodes[ne]),list(nodes[se])]}}]}
        (project_root/"inputs"/"osm_east_reference_WGS84.geojson").write_text(json.dumps(gj,indent=2),encoding="utf-8")
    except Exception as e:
        record.update(status="FETCH_FAILED",error=str(e))
    (project_root/"outputs"/"osm_fetch_optional_record.json").write_text(json.dumps(record,indent=2,ensure_ascii=False),encoding="utf-8")
    return record


### 1 · Setup and configuration
The configuration is written from the **EDIT PARAMETERS HERE** cell above. Reviewers should first run the unchanged baseline. For an optional experiment, save a copy, change one parameter only, set `RUN_MODE = "experiment"`, and rerun all cells. The six placements remain alternatives, not six simultaneous seats. `parallel` means the long seat axis is parallel to the east reference edge (north–south in the local drawing).


In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / 'inputs' / 'config.json').exists():
    raise FileNotFoundError('Run the EDIT PARAMETERS HERE cell first.')
cfg, variants = load_inputs(ROOT)
print('Study:', cfg['study'])
print('Mode:', RUN_MODE)
print('Window:', cfg['window_width_m'], '×', cfg['window_height_m'], 'm')
print('Assumed widths:', cfg['corridor_half_widths_m'], '| edges:', cfg['flow_edge_proxies'])
print('Proposed band c1 / c2:', cfg['band_inner_offset_c1_m'], cfg['band_outer_offset_c2_m'], 'm')
print('Assumed threshold s:', cfg['assumed_band_threshold_pct'], '%')
display(variants)


### 2 · Geometric input and method
The local schematic frame uses `x=0` for the east boundary and `y=0` for the north boundary. A candidate at inward distance `u` has `x=-u`. We calculate:

- `corridor_overlap_m2 = Area(seat ∩ buffer(proxy edge, w))`
- `proposed_band_share_pct = 100 × Area(seat ∩ [buffer(edge,w+c₂) − buffer(edge,w+c₁)]) / Area(seat)`
- All buffered geometries are clipped to Window C. **0 overlap means no *modelled* conflict only.**

The proposed `c₁=0.30 m`, `c₂=1.50 m` and `s=60%` are **researcher assumptions**, not validated short-stay comfort standards. The garden boundary is not shown to be a walking path.


In [ ]:
wpoly = window_polygon(cfg)
print('Window C schematic area:', wpoly.area, 'm²')
print('S1 seat geometry (LOCAL metres):', seat_polygon(variants.iloc[0], cfg).wkt)
print('S1 base east geometry metrics:', {k:v for k,v in overlay_metrics(seat_polygon(variants.iloc[0],cfg),cfg,'east',1.0).items() if isinstance(v,(int,float))})

#### GeoPandas independent spatial overlay demonstration
The reusable scenario engine uses Shapely intersections, while this cell performs an **actual GeoPandas overlay** for the S2, `w=1.50 m` known case and verifies that the intersected area is **0.30 m²**. Both input layers have a **local metre** frame without a claimed EPSG CRS; do **not** assign the Hong Kong Grid EPSG:2326 to these synthetic coordinates.

In [ ]:
import geopandas as gpd

seats_gdf = gpd.GeoDataFrame(
    variants[['variant','orientation']].copy(),
    geometry=[seat_polygon(v, cfg) for _, v in variants.iterrows()],
    crs=None  # schematic local metres; NOT EPSG:2326
)
corridor_geom = proxy_line(cfg, 'east').buffer(1.5, cap_style='flat').intersection(window_polygon(cfg))
corridor_gdf = gpd.GeoDataFrame({'proxy': ['east corridor w=1.5m']}, geometry=[corridor_geom], crs=None)
intersections_gdf = gpd.overlay(seats_gdf, corridor_gdf, how='intersection')
intersections_gdf['overlap_m2'] = intersections_gdf.geometry.area
print('GeoPandas overlay intersections (w=1.5m, east proxy):')
display(intersections_gdf[['variant','orientation','overlap_m2']])
assert len(intersections_gdf)==1 and intersections_gdf.variant.iloc[0]=='S2'
assert abs(float(intersections_gdf.overlap_m2.iloc[0])-0.30)<1e-8
print('PASS: the independent GeoPandas overlay agrees with Shapely and 1D arithmetic.')

### 3 · Generate six alternatives × three corridor widths × two flow-edge proxies
The same seat area and evaluation functions are used for every scenario. A row labelled `ASSUMED_PROXY_PASS` means only that the *hypothetical* clearance and researcher-chosen band-share rule pass under that scenario.


In [ ]:
results = generate_scenarios(cfg, variants)
print('Row count:',len(results),'(expected 6 × 3 × 2 = 36)')
display(results[['variant','orientation','flow_edge_proxy','corridor_half_width_w_m',
                 'corridor_overlap_m2','proposed_band_share_pct','scenario_status']].head(12))

### 4 · Verify the calculation before interpreting any design
In **baseline** mode, the notebook runs the original 25 checks: seven known-case / structural assertions plus eighteen independent one-dimensional interval cross-checks for the east proxy. In **experiment** mode, the fixed baseline values are no longer expected; the notebook instead checks scenario count, seat area and the eighteen interval calculations using the changed inputs.


In [ ]:
if RUN_MODE == "baseline":
    check_df = verification_checks(cfg, variants, results)
    print(f"Baseline internal assertions: {(check_df.status == 'PASS').sum()}/{len(check_df)} PASS")
else:
    # Parameter-independent structure checks + interval cross-checks for the edited inputs.
    checks = []
    def add_check(name, expected, observed, tol=1e-8):
        ok = abs(float(observed) - float(expected)) <= tol
        checks.append({"check": name, "expected": expected, "observed": observed,
                       "status": "PASS" if ok else "FAIL"})
        if not ok:
            raise AssertionError(f"{name}: expected {expected}, observed {observed}")
    add_check("Scenario row count", 36, len(results))
    add_check("Minimum seat area = 2 m2", 2.0, results.seat_area_m2.min())
    add_check("Maximum seat area = 2 m2", 2.0, results.seat_area_m2.max())
    for _, v in variants.iterrows():
        for w in cfg["corridor_half_widths_m"]:
            row = results[(results.variant == v.variant) &
                          (results.flow_edge_proxy == "east") &
                          (np.isclose(results.corridor_half_width_w_m, float(w)))]
            if len(row) != 1:
                raise AssertionError(f"Expected one row for {v.variant}/east/{w}")
            expected = manual_band_pct_east(v, cfg, float(w))
            observed = float(row.iloc[0].proposed_band_share_pct)
            add_check(f"Edited-input interval check {v.variant}, east w={w}", expected, observed, 1e-6)
    check_df = pd.DataFrame(checks)
    print(f"Experiment structural + interval checks: {(check_df.status == 'PASS').sum()}/{len(check_df)} PASS")

display(check_df)


### 5 · Export auditable CSV and figures, then inspect the results
One command invokes the complete set of processing steps and records actually used library versions to `outputs/a3_run_record.json`.


In [ ]:
artifacts = run_project(ROOT, perform_checks=False)
summary = artifacts['summary']
# Save the checks from the cell above (baseline or experiment mode).
check_df.to_csv(ROOT / 'outputs' / 'a3_verification_internal.csv', index=False)
print('Output files written to:', ROOT/'outputs')
display(summary)
print('All six original alternatives pass all tested widths AND both flow proxies:', bool(summary.all_six_cases_both_edges_pass.any()))
print('S1 is the BASE-CASE leader, but not robust under all tested assumptions.')


In [ ]:
display(Image(filename=str(ROOT/'outputs'/'a3_six_alternative_placements.png'),width=960))

In [ ]:
display(Image(filename=str(ROOT/'outputs'/'a3_base_comparison.png'),width=880))

### 6 · Threshold / parameter sensitivity (Week 9: draw the line)
The baseline result is: S1 passes the assumed **s=60%** band rule on the east proxy for approximately `w=0.80–1.70 m`, before corridor overlap occurs. This text describes the unchanged baseline only. If you edit parameters, read the newly generated plot and CSV; the plot recalculates the feasible range from the current inputs.


In [ ]:
display(Image(filename=str(ROOT/'outputs'/'a3_w_sensitivity.png'),width=880))

### 7 · Optional generative improvement: sweep the seat centre by 0.05 m
The position sweep is an exploratory design iteration, not the main research claim. With the **east proxy only**, it searches parallel-seat centre positions while retaining the tested corridor widths and thresholds. The baseline candidate range is 2.00–2.15 m. In experiment mode, inspect the new table rather than expecting the baseline range.


In [ ]:
sweep = artifacts['sweep']
passing = sweep[sweep.passes_all_east_widths_assumed]
display(passing)
if RUN_MODE == "baseline":
    assert len(passing) == 4 and abs(passing.centre_u_m.min()-2.00) < 1e-8 and abs(passing.centre_u_m.max()-2.15) < 1e-8
    print('Baseline exploratory east-only sweep reproduced: four centres at 0.05 m intervals.')
else:
    print('Experiment sweep generated for edited parameters; compare its range with the baseline.')


### 8 · Optional OSM verification (**disabled by default**)
The package can **optionally** fetch full OSM way 27059126 when internet access is available. It checks NE/SE node IDs and computes their distance in EPSG:2326, saving provenance. This operation does **not** substitute the schematic local path proxy with an observed walking route. If blocked, record `FETCH_FAILED` honestly.


In [ ]:
FETCH_OSM_ONLINE = False  # flip to True only if you want a live provenance check
osm_record = osm_fetch_optional(ROOT, do_fetch=FETCH_OSM_ONLINE)
print(osm_record)

### 9 · A3 interpretation / transparent limits

| Verified here | Not established here |
|---|---|
| 36 scenarios generated | Actual ground-level pedestrian route adjacent to C |
| 2.00 m² area of each seat | Measured dimensions and true boundary of C |
| Analytic S1=80%, S1 at w=0.75m=55% | Flow intensity, short-stay count or dwell duration |
| S2 overlaps 0.30m² at w=1.5m | Behavioural validity of c₁/c₂/s |
| Internal assertions / script rerun | Independent human QGIS rerun |

**Decision:** retain S1 as *a conditional base-case benchmark*, not a final sitting-location recommendation. No original variant robustly passes the whole assumed test. The next evidence task is to verify C's local walking edges and measure real short-stay activity before asserting comfort or design suitability.

**AI-assisted code note:** this notebook, script and figures were drafted with AI assistance and actually executed, with independent arithmetic cross-checks in the package. An external/manual spatial validation has **not** been recorded.
